# 17 · Reviewed catalog evidence

**Goal:** create a small fictional, locally reviewed evidence bundle, connect it to the HTTP application, and confirm that the canonical item data does not change. The example source URLs are placeholders. No network access, license verification, or image download occurs.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from fastapi.testclient import TestClient
from savoroute.data.catalog import load_catalog
from savoroute.evidence.review import catalog_digest, prepare_review, publish_evidence
from savoroute.api.app import create_app
catalog = load_catalog()
print('Catalog digest:', catalog_digest(catalog))

In [ ]:
with TemporaryDirectory() as folder:
    folder = Path(folder)
    proposals = folder / 'proposals.json'
    review = folder / 'review.json'
    approved = folder / 'approved.json'
    sample = {'schema_version': 1, 'catalog_digest': catalog_digest(catalog), 'claims': [{
        'claim_id': 'sample_menu', 'item_id': 'r001', 'kind': 'menu_url',
        'value': 'https://example.org/fictional-menu',
        'source_url': 'https://example.org/fictional-restaurant',
        'observed_on': '2026-11-12', 'license': None, 'attribution': None,
    }]}
    proposals.write_text(json.dumps(sample))
    assert prepare_review(catalog, proposals, review) == 1
    sheet = json.loads(review.read_text())
    # This approval is a synthetic workflow demonstration, not a true validation.
    sheet['decisions'][0].update({
        'decision': 'approve', 'reviewer': 'fictional-reviewer',
        'reason': 'workflow demonstration only', 'reviewed_on': '2026-11-12',
    })
    review.write_text(json.dumps(sheet))
    publish_evidence(catalog, proposals, review, approved)
    with TestClient(create_app(evidence_path=approved)) as client:
        original = client.get('/v1/items/r001').json()
        evidence = client.get('/v1/items/r001/evidence').json()
        assert 'reviewed_claims' not in original
        assert evidence['reviewed_claims'][0]['kind'] == 'menu_url'
        print(original['title'], '| reviewed references:', len(evidence['reviewed_claims']))

## Interpretation

The reference remains separate from the catalog and does not influence BM25, RRF or recommendation ranking. It is only an accountable reviewed claim; the demonstration makes no assertion about restaurant menus, image rights, availability, or user visits.